In [0]:

# ============================================================================
#  Install Libraries
# ============================================================================

# Install required packages
%pip install xgboost shap scikit-learn plotly kaleido

# Restart Python
dbutils.library.restartPython()

In [0]:
# ============================================================================
# Import Libraries
# ============================================================================

# Standard libraries
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import random

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Machine Learning
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, 
    roc_auc_score, 
    confusion_matrix,
    roc_curve
)
import shap

# MLflow
import mlflow
import mlflow.xgboost

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

print("="*60)
print("LIBRARIES IMPORTED SUCCESSFULLY")
print("="*60)
print(f"✓ All libraries imported")
print(f"✓ Spark version: {spark.version}")
print("="*60)

print("\n✓ MLflow available - experiment will be set during model training")
print(f"✓ MLflow version: {mlflow.__version__}")
print("="*60)
print("\n✓ SEPSIS EWS SYSTEM - ENVIRONMENT READY")
print("="*60)

In [0]:

# ============================================================================
# Database Setup
# ============================================================================

# Use existing healthcare_monitoring database
spark.sql("CREATE DATABASE IF NOT EXISTS healthcare_monitoring")
spark.sql("USE healthcare_monitoring")

print("✓ Using database: healthcare_monitoring\n")

# List tables
tables = spark.sql("SHOW TABLES").toPandas()
print(f"Existing tables: {len(tables)}")
if len(tables) > 0:
    print(tables[['tableName']])


In [0]:


# ============================================================================
# MIMIC-Style ICU Data Simulator
# ============================================================================

class MIMICDataSimulator:
    """
    Simulates MIMIC-III style ICU data for sepsis prediction.
    
    Generates realistic patient stays with:
    - Hourly vital signs
    - Periodic lab results (every 6 hours)
    - 20% of patients develop sepsis
    - Progressive deterioration 6 hours before sepsis onset
    """
    
    def __init__(self, patient_id, develops_sepsis=False):
        """
        Initialize simulator for one patient ICU stay.
        
        Args:
            patient_id (str): Unique patient identifier
            develops_sepsis (bool): Whether patient develops sepsis
        """
        self.patient_id = patient_id
        self.develops_sepsis = develops_sepsis
        
        # Patient baseline vitals (individual variation)
        self.baseline = {
            'heart_rate': np.random.normal(75, 10),
            'systolic_bp': np.random.normal(120, 15),
            'diastolic_bp': np.random.normal(80, 10),
            'spo2': np.random.normal(97, 2),
            'respiratory_rate': np.random.normal(16, 3),
            'temperature': np.random.normal(37.0, 0.3)
        }
        
        # Lab baseline values
        self.lab_baseline = {
            'wbc': np.random.normal(7.5, 2.0),
            'hemoglobin': np.random.normal(13.5, 1.5),
            'platelets': np.random.normal(250, 50),
            'creatinine': np.random.normal(1.0, 0.2),
            'lactate': np.random.normal(1.2, 0.3)
        }
        
        # If develops sepsis, determine onset time (between 12-18 hours)
        if develops_sepsis:
            self.sepsis_onset_hour = np.random.randint(12, 19)
        else:
            self.sepsis_onset_hour = None
    
    def generate_vitals(self, hour):
        """
        Generate vital signs for specific hour.
        
        Args:
            hour (int): Hour since ICU admission (0-23)
            
        Returns:
            dict: Vital signs with realistic variation
        """
        
        # Calculate hours until sepsis (if applicable)
        if self.develops_sepsis and self.sepsis_onset_hour is not None:
            hours_to_sepsis = self.sepsis_onset_hour - hour
        else:
            hours_to_sepsis = 999  # Large number (no sepsis)
        
        # Normal variation
        vitals = {
            'heart_rate': self.baseline['heart_rate'] + np.random.normal(0, 5),
            'systolic_bp': self.baseline['systolic_bp'] + np.random.normal(0, 8),
            'diastolic_bp': self.baseline['diastolic_bp'] + np.random.normal(0, 5),
            'spo2': self.baseline['spo2'] + np.random.normal(0, 1),
            'respiratory_rate': self.baseline['respiratory_rate'] + np.random.normal(0, 2),
            'temperature': self.baseline['temperature'] + np.random.normal(0, 0.2)
        }
        
        # Progressive deterioration if approaching sepsis (6 hours before)
        if self.develops_sepsis and 0 < hours_to_sepsis <= 6:
            # Severity increases as sepsis approaches
            severity = 1 - (hours_to_sepsis / 6)  # 0 to 1
            
            # Tachycardia (increasing heart rate)
            vitals['heart_rate'] += severity * 30
            
            # Hypotension (decreasing blood pressure)
            vitals['systolic_bp'] -= severity * 25
            vitals['diastolic_bp'] -= severity * 15
            
            # Tachypnea (increasing respiratory rate)
            vitals['respiratory_rate'] += severity * 8
            
            # Fever (increasing temperature)
            vitals['temperature'] += severity * 1.5
            
            # Mild hypoxia
            vitals['spo2'] -= severity * 4
        
        # At sepsis onset and after, maintain abnormal values
        elif self.develops_sepsis and hours_to_sepsis <= 0:
            vitals['heart_rate'] += 35
            vitals['systolic_bp'] -= 30
            vitals['diastolic_bp'] -= 20
            vitals['respiratory_rate'] += 10
            vitals['temperature'] += 1.8
            vitals['spo2'] -= 5
        
        # Clip to physiological ranges
        vitals['heart_rate'] = np.clip(vitals['heart_rate'], 40, 160)
        vitals['systolic_bp'] = np.clip(vitals['systolic_bp'], 60, 200)
        vitals['diastolic_bp'] = np.clip(vitals['diastolic_bp'], 30, 120)
        vitals['spo2'] = np.clip(vitals['spo2'], 85, 100)
        vitals['respiratory_rate'] = np.clip(vitals['respiratory_rate'], 8, 40)
        vitals['temperature'] = np.clip(vitals['temperature'], 35.0, 40.0)
        
        return vitals
    
    def generate_labs(self, hour):
        """
        Generate lab results (every 6 hours).
        
        Args:
            hour (int): Hour since ICU admission
            
        Returns:
            dict: Lab values or None if not lab time
        """
        
        # Labs drawn every 6 hours (0, 6, 12, 18)
        if hour % 6 != 0:
            return None
        
        # Calculate hours until sepsis
        if self.develops_sepsis and self.sepsis_onset_hour is not None:
            hours_to_sepsis = self.sepsis_onset_hour - hour
        else:
            hours_to_sepsis = 999
        
        # Normal variation
        labs = {
            'wbc': self.lab_baseline['wbc'] + np.random.normal(0, 1.5),
            'hemoglobin': self.lab_baseline['hemoglobin'] + np.random.normal(0, 0.8),
            'platelets': self.lab_baseline['platelets'] + np.random.normal(0, 30),
            'creatinine': self.lab_baseline['creatinine'] + np.random.normal(0, 0.1),
            'lactate': self.lab_baseline['lactate'] + np.random.normal(0, 0.2)
        }
        
        # Abnormal labs if approaching sepsis
        if self.develops_sepsis and 0 < hours_to_sepsis <= 6:
            severity = 1 - (hours_to_sepsis / 6)
            
            # Leukocytosis or leukopenia
            if np.random.random() > 0.5:
                labs['wbc'] += severity * 8  # Leukocytosis
            else:
                labs['wbc'] -= severity * 4  # Leukopenia
            
            # Elevated lactate (tissue hypoperfusion)
            labs['lactate'] += severity * 3.5
            
            # Thrombocytopenia
            labs['platelets'] -= severity * 100
            
            # Acute kidney injury
            labs['creatinine'] += severity * 1.0
        
        elif self.develops_sepsis and hours_to_sepsis <= 0:
            labs['wbc'] += 9 if np.random.random() > 0.3 else -5
            labs['lactate'] += 4.0
            labs['platelets'] -= 120
            labs['creatinine'] += 1.2
        
        # Clip to physiological ranges
        labs['wbc'] = np.clip(labs['wbc'], 1.0, 30.0)
        labs['hemoglobin'] = np.clip(labs['hemoglobin'], 6.0, 18.0)
        labs['platelets'] = np.clip(labs['platelets'], 20, 500)
        labs['creatinine'] = np.clip(labs['creatinine'], 0.5, 5.0)
        labs['lactate'] = np.clip(labs['lactate'], 0.5, 15.0)
        
        return labs
    
    def generate_icu_stay(self, hours=24):
        """
        Generate complete ICU stay data.
        
        Args:
            hours (int): Length of stay in hours
            
        Returns:
            pd.DataFrame: Complete patient timeline
        """
        
        records = []
        
        # Most recent lab values (forward-fill)
        current_labs = None
        
        for hour in range(hours):
            # Timestamp
            timestamp = datetime.now() - timedelta(hours=hours-hour)
            
            # Generate vitals (every hour)
            vitals = self.generate_vitals(hour)
            
            # Generate labs (every 6 hours)
            new_labs = self.generate_labs(hour)
            if new_labs is not None:
                current_labs = new_labs
            
            # Determine label (sepsis in next 6 hours?)
            if self.develops_sepsis and self.sepsis_onset_hour is not None:
                hours_to_sepsis = self.sepsis_onset_hour - hour
                # Label as positive if sepsis onset within 6 hours
                sepsis_in_6h = 1 if 0 < hours_to_sepsis <= 6 else 0
                has_sepsis = 1 if hours_to_sepsis <= 0 else 0
            else:
                sepsis_in_6h = 0
                has_sepsis = 0
            
            # Combine into record
            record = {
                'patient_id': self.patient_id,
                'hour': hour,
                'timestamp': timestamp,
                **vitals,
                'sepsis_in_6h': sepsis_in_6h,  # Target variable
                'has_sepsis_now': has_sepsis
            }
            
            # Add labs (forward-filled from most recent)
            if current_labs is not None:
                record.update(current_labs)
            else:
                # Use baseline if no labs yet
                record.update(self.lab_baseline)
            
            records.append(record)
        
        return pd.DataFrame(records)

# Test the simulator
print("Testing MIMIC-Style ICU Data Simulator...")
print("="*60)

# Test with non-sepsis patient
test_no_sepsis = MIMICDataSimulator("TEST_NO_SEPSIS", develops_sepsis=False)
test_data_no = test_no_sepsis.generate_icu_stay(24)

print(f"\n✓ Generated non-sepsis patient: {len(test_data_no)} hours")
print(f"  Sepsis cases: {test_data_no['sepsis_in_6h'].sum()}")

# Test with sepsis patient
test_sepsis = MIMICDataSimulator("TEST_SEPSIS", develops_sepsis=True)
test_data_yes = test_sepsis.generate_icu_stay(24)

print(f"\n✓ Generated sepsis patient: {len(test_data_yes)} hours")
print(f"  Sepsis onset hour: {test_sepsis.sepsis_onset_hour}")
print(f"  Hours labeled as 'sepsis in 6h': {test_data_yes['sepsis_in_6h'].sum()}")

print("\nSample data (sepsis patient, hours 10-15):")
cols = ['hour', 'heart_rate', 'systolic_bp', 'temperature', 'lactate', 'sepsis_in_6h']
print(test_data_yes[cols].iloc[10:16])

print("\n" + "="*60)
print("✓ MIMIC Simulator validated and ready")
print("="*60)



In [0]:

# ============================================================================
# Generate Full Dataset (1,000 Patients)
# ============================================================================

print("Generating MIMIC-style ICU dataset...")
print("="*60)
print("Configuration:")
print("  - Total patients: 1,000")
print("  - Sepsis rate: 20% (200 patients)")
print("  - Hours per patient: 24")
print("  - Total observations: 24,000")
print("\nThis will take 2-3 minutes...\n")

# Configuration
num_patients = 1000
sepsis_rate = 0.20  # 20% develop sepsis
hours_per_patient = 24

# Track which patients develop sepsis
num_sepsis = int(num_patients * sepsis_rate)
num_no_sepsis = num_patients - num_sepsis

all_patient_data = []

# Generate sepsis patients
print(f"Generating {num_sepsis} sepsis patients...")
for i in range(num_sepsis):
    patient_id = f"SP{i+1:04d}"  # SP0001, SP0002, etc.
    
    simulator = MIMICDataSimulator(patient_id, develops_sepsis=True)
    patient_df = simulator.generate_icu_stay(hours=hours_per_patient)
    all_patient_data.append(patient_df)
    
    if (i + 1) % 50 == 0:
        print(f"  ✓ Generated {i + 1}/{num_sepsis} sepsis patients...")

# Generate non-sepsis patients
print(f"\nGenerating {num_no_sepsis} non-sepsis patients...")
for i in range(num_no_sepsis):
    patient_id = f"NP{i+1:04d}"  # NP0001, NP0002, etc.
    
    simulator = MIMICDataSimulator(patient_id, develops_sepsis=False)
    patient_df = simulator.generate_icu_stay(hours=hours_per_patient)
    all_patient_data.append(patient_df)
    
    if (i + 1) % 100 == 0:
        print(f"  ✓ Generated {i + 1}/{num_no_sepsis} non-sepsis patients...")

# Combine all data
combined_df = pd.concat(all_patient_data, ignore_index=True)

print(f"\n{'='*60}")
print("DATASET GENERATION COMPLETE")
print(f"{'='*60}")
print(f"Total patients: {num_patients}")
print(f"Total observations: {len(combined_df):,}")
print(f"Sepsis patients: {num_sepsis} ({100*sepsis_rate:.0f}%)")
print(f"Non-sepsis patients: {num_no_sepsis} ({100*(1-sepsis_rate):.0f}%)")

# Label distribution
print(f"\nTarget variable distribution:")
print(f"  Sepsis within 6h (positive): {combined_df['sepsis_in_6h'].sum():,} ({100*combined_df['sepsis_in_6h'].mean():.1f}%)")
print(f"  No sepsis (negative): {(~combined_df['sepsis_in_6h'].astype(bool)).sum():,} ({100*(1-combined_df['sepsis_in_6h'].mean()):.1f}%)")

print(f"\nData shape: {combined_df.shape}")
print(f"Columns: {list(combined_df.columns)}")

# Show sample
print(f"\nSample data (first 5 rows):")
display_cols = ['patient_id', 'hour', 'heart_rate', 'systolic_bp', 'lactate', 'sepsis_in_6h']
print(combined_df[display_cols].head())

print(f"\n{'='*60}")
print("✓ Dataset ready for feature engineering")
print(f"{'='*60}")

In [0]:
# ============================================================================
# Feature Engineering, Model Training, and Evaluation
# ============================================================================

# ============================================================================
# Create Delta Table and Load Data
# ============================================================================

print("Creating Delta table for ICU data...")

# Drop existing table
spark.sql("DROP TABLE IF EXISTS healthcare_monitoring.icu_sepsis_data")

# Convert to Spark DataFrame
spark_df = spark.createDataFrame(combined_df)

# Write to Delta table
spark_df.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_monitoring.icu_sepsis_data"
)

# Verify
record_count = spark.table("healthcare_monitoring.icu_sepsis_data").count()
print(f"✓ Loaded {record_count:,} records to icu_sepsis_data table")

# Show sample
print("\nSample from Delta table:")
display(spark.table("healthcare_monitoring.icu_sepsis_data").limit(5))

print("\n" + "="*60)
print("✓ Data loaded to Delta Lake")
print("="*60)


In [0]:
# ============================================================================
# Time-Series Feature Engineering
# ============================================================================

print("Engineering time-series features...")
print("="*60)

# Read data
icu_df = spark.table("healthcare_monitoring.icu_sepsis_data")

# Define window specifications for time-series features
# 3-hour window
window_3h = (Window
    .partitionBy("patient_id")
    .orderBy("hour")
    .rowsBetween(-2, 0)  # Current + 2 previous hours
)

# 6-hour window
window_6h = (Window
    .partitionBy("patient_id")
    .orderBy("hour")
    .rowsBetween(-5, 0)  # Current + 5 previous hours
)

# Lag window (for trends)
window_lag = (Window
    .partitionBy("patient_id")
    .orderBy("hour")
)

print("Creating features:")
print("  - Rolling means (3h, 6h windows)")
print("  - Rolling standard deviations (variability)")
print("  - Trends (slopes, changes)")
print("  - Derived clinical metrics")
print()

# Feature engineering
featured_df = (icu_df
    # ========================================================================
    # ROLLING STATISTICS (3-hour windows)
    # ========================================================================
    .withColumn("hr_mean_3h", F.avg("heart_rate").over(window_3h))
    .withColumn("hr_std_3h", F.stddev("heart_rate").over(window_3h))
    .withColumn("sbp_mean_3h", F.avg("systolic_bp").over(window_3h))
    .withColumn("temp_mean_3h", F.avg("temperature").over(window_3h))
    .withColumn("rr_mean_3h", F.avg("respiratory_rate").over(window_3h))
    
    # ========================================================================
    # ROLLING STATISTICS (6-hour windows)
    # ========================================================================
    .withColumn("hr_mean_6h", F.avg("heart_rate").over(window_6h))
    .withColumn("hr_std_6h", F.stddev("heart_rate").over(window_6h))
    .withColumn("sbp_min_6h", F.min("systolic_bp").over(window_6h))
    .withColumn("spo2_min_6h", F.min("spo2").over(window_6h))
    
    # ========================================================================
    # TRENDS (change from previous hour)
    # ========================================================================
    .withColumn("hr_prev", F.lag("heart_rate", 1).over(window_lag))
    .withColumn("temp_prev", F.lag("temperature", 1).over(window_lag))
    
    .withColumn("hr_change", F.col("heart_rate") - F.col("hr_prev"))
    .withColumn("temp_change", F.col("temperature") - F.col("temp_prev"))
    
    # ========================================================================
    # DERIVED CLINICAL METRICS
    # ========================================================================
    # Mean Arterial Pressure
    .withColumn("mean_arterial_pressure", 
        F.col("diastolic_bp") + (F.col("systolic_bp") - F.col("diastolic_bp")) / 3
    )
    
    # Shock Index (HR/SBP - early shock indicator)
    .withColumn("shock_index", 
        F.col("heart_rate") / F.col("systolic_bp")
    )
    
    # qSOFA Score (Quick Sepsis-related Organ Failure Assessment)
    .withColumn("qsofa_rr", F.when(F.col("respiratory_rate") >= 22, 1).otherwise(0))
    .withColumn("qsofa_sbp", F.when(F.col("systolic_bp") <= 100, 1).otherwise(0))
    .withColumn("qsofa_score", F.col("qsofa_rr") + F.col("qsofa_sbp"))
)

# ============================================================================
# FILL NULL VALUES
# ============================================================================

print("Handling null values from window operations...")

# For standard deviations (null for first few hours), fill with 0
featured_df = (featured_df
    .fillna(0, subset=["hr_std_3h", "hr_std_6h", "hr_change", "temp_change"])
)

# For rolling means, use coalesce to fill with current value if null
featured_df = (featured_df
    .withColumn("hr_mean_3h", F.coalesce(F.col("hr_mean_3h"), F.col("heart_rate")))
    .withColumn("hr_mean_6h", F.coalesce(F.col("hr_mean_6h"), F.col("heart_rate")))
    .withColumn("sbp_mean_3h", F.coalesce(F.col("sbp_mean_3h"), F.col("systolic_bp")))
    .withColumn("temp_mean_3h", F.coalesce(F.col("temp_mean_3h"), F.col("temperature")))
    .withColumn("rr_mean_3h", F.coalesce(F.col("rr_mean_3h"), F.col("respiratory_rate")))
    .withColumn("sbp_min_6h", F.coalesce(F.col("sbp_min_6h"), F.col("systolic_bp")))
    .withColumn("spo2_min_6h", F.coalesce(F.col("spo2_min_6h"), F.col("spo2")))
)

print("✓ Null values handled")

# ============================================================================
# SAVE TO DELTA TABLE
# ============================================================================

# Save to new table
featured_df.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_monitoring.icu_sepsis_features"
)

feature_count = spark.table("healthcare_monitoring.icu_sepsis_features").count()
print(f"\n✓ Created feature table with {feature_count:,} records")

# Show feature summary
print("\nFeature Statistics:")
feature_stats = spark.sql("""
    SELECT 
        ROUND(AVG(hr_mean_6h), 1) as avg_hr_6h,
        ROUND(AVG(hr_std_6h), 2) as avg_hr_std_6h,
        ROUND(AVG(shock_index), 2) as avg_shock_index,
        ROUND(AVG(lactate), 2) as avg_lactate,
        ROUND(AVG(qsofa_score), 2) as avg_qsofa
    FROM healthcare_monitoring.icu_sepsis_features
""").toPandas()

print(feature_stats.to_string(index=False))

# Show sample with features
print("\nSample records with engineered features:")
display(spark.table("healthcare_monitoring.icu_sepsis_features")
    .select(
        "patient_id", "hour", "heart_rate", "hr_mean_6h", "hr_std_6h",
        "shock_index", "lactate", "sepsis_in_6h"
    )
    .limit(10)
)

print("\n" + "="*60)
print("✓ Feature engineering complete - 20 features created")
print("="*60)

In [0]:

# ============================================================================
# Prepare Training Data
# ============================================================================

print("Preparing training data...")
print("="*60)

# Load featured data
featured_data = spark.table("healthcare_monitoring.icu_sepsis_features").toPandas()

print(f"Total observations: {len(featured_data):,}")
print(f"Positive class (sepsis in 6h): {featured_data['sepsis_in_6h'].sum():,}")
print(f"Negative class: {(~featured_data['sepsis_in_6h'].astype(bool)).sum():,}")
print(f"Class imbalance ratio: {(~featured_data['sepsis_in_6h'].astype(bool)).sum() / featured_data['sepsis_in_6h'].sum():.1f}:1")

# Select features for model
feature_columns = [
    # Raw vital signs
    'heart_rate', 'systolic_bp', 'diastolic_bp', 'spo2', 
    'respiratory_rate', 'temperature',
    
    # Rolling statistics
    'hr_mean_3h', 'hr_std_3h', 'hr_mean_6h', 'hr_std_6h',
    'sbp_mean_3h', 'sbp_min_6h', 'spo2_min_6h',
    'temp_mean_3h', 'rr_mean_3h',
    
    # Trends
    'hr_change', 'temp_change',
    
    # Derived metrics
    'mean_arterial_pressure', 'shock_index', 'qsofa_score',
    
    # Labs
    'wbc', 'lactate', 'creatinine', 'platelets'
]

print(f"\nFeatures selected: {len(feature_columns)}")
print("Feature list:")
for i, feat in enumerate(feature_columns, 1):
    print(f"  {i:2d}. {feat}")

# Create feature matrix and target
X = featured_data[feature_columns]
y = featured_data['sepsis_in_6h']

# Train-test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n{'='*60}")
print("TRAIN-TEST SPLIT")
print(f"{'='*60}")
print(f"Training set:   {len(X_train):,} samples")
print(f"  - Positive:   {y_train.sum():,} ({100*y_train.mean():.1f}%)")
print(f"  - Negative:   {(~y_train.astype(bool)).sum():,}")
print(f"\nTest set:       {len(X_test):,} samples")
print(f"  - Positive:   {y_test.sum():,} ({100*y_test.mean():.1f}%)")
print(f"  - Negative:   {(~y_test.astype(bool)).sum():,}")
print(f"{'='*60}")

print("\n✓ Training data prepared")



In [0]:
# ============================================================================
# Train XGBoost Model
# ============================================================================

print("="*60)
print("TRAINING XGBOOST MODEL")
print("="*60)

# ============================================================================
# Model Training 
# ============================================================================

# Calculate scale_pos_weight for class imbalance
scale_pos_weight = (~y_train.astype(bool)).sum() / y_train.sum()

print(f"\nClass imbalance handling:")
print(f"  Scale pos weight: {scale_pos_weight:.2f}")
print(f"  (Ratio of negative to positive samples)")

# XGBoost parameters
params = {
    'max_depth': 6,
    'learning_rate': 0.1,
    'n_estimators': 100,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'scale_pos_weight': scale_pos_weight,
    'random_state': 42,
    'tree_method': 'hist'
}

print(f"\nModel configuration:")
for key, value in params.items():
    print(f"  {key}: {value}")

# Start MLflow run (using default experiment)
with mlflow.start_run(run_name="xgboost_sepsis_ews") as run:
    
    print(f"\n{'='*60}")
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"{'='*60}")
    
    # Log parameters
    mlflow.log_params(params)
    mlflow.log_param("n_features", len(feature_columns))
    mlflow.log_param("train_samples", len(X_train))
    mlflow.log_param("test_samples", len(X_test))
    
    # Train model
    print("\nTraining XGBoost model...")
    print("This may take 2-3 minutes...\n")
    
    model = xgb.XGBClassifier(**params)
    
    # Train with evaluation set
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )
    
    print("✓ Training complete!")
    
    # ========================================================================
    # EVALUATION
    # ========================================================================
    
    print(f"\n{'='*60}")
    print("MODEL EVALUATION")
    print(f"{'='*60}")
    
    # Predictions
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)
    
    # ROC-AUC
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Classification report
    report = classification_report(y_test, y_pred, output_dict=True)
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Log metrics
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("accuracy", report['accuracy'])
    mlflow.log_metric("precision", report['1']['precision'])
    mlflow.log_metric("recall", report['1']['recall'])
    mlflow.log_metric("f1_score", report['1']['f1-score'])
    
    # Print results
    print(f"\nROC-AUC:   {roc_auc:.3f}")
    print(f"Accuracy:  {report['accuracy']:.3f}")
    print(f"Precision: {report['1']['precision']:.3f}")
    print(f"Recall:    {report['1']['recall']:.3f}")
    print(f"F1-Score:  {report['1']['f1-score']:.3f}")
    
    print(f"\nConfusion Matrix:")
    print(f"                 Predicted")
    print(f"                 No Sepsis  Sepsis")
    print(f"Actual No Sepsis {cm[0,0]:9d}  {cm[0,1]:6d}")
    print(f"       Sepsis    {cm[1,0]:9d}  {cm[1,1]:6d}")
    
    # Feature importance
    feature_importance = pd.DataFrame({
        'feature': feature_columns,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)
    
    print(f"\n{'='*60}")
    print("TOP 10 MOST IMPORTANT FEATURES")
    print(f"{'='*60}")
    for idx, row in feature_importance.head(10).iterrows():
        print(f"  {row['feature']:30s} {row['importance']:.4f}")
    
    # Log model
    print(f"\n{'='*60}")
    print("Logging model to MLflow...")
    
    try:
        # Try to log model
        mlflow.xgboost.log_model(model, "model")
        print("✓ Model logged to MLflow")
    except Exception as e:
        print(f"⚠ Model logging skipped: {e}")
        print("✓ Training complete (model in memory)")
    
    # Log feature importance as artifact
    try:
        mlflow.log_dict(feature_importance.to_dict(), "feature_importance.json")
        print("✓ Feature importance logged")
    except Exception as e:
        print(f"⚠ Artifact logging skipped: {e}")
    
    print(f"{'='*60}")
    
    run_id = run.info.run_id

print(f"\n{'='*60}")
print("✓ TRAINING COMPLETE!")
print(f"{'='*60}")
print(f"MLflow Run ID: {run_id}")
print("\nTo view in MLflow UI:")
print("  1. Click 'Experiments' in left sidebar")
print("  2. Look for your run (sorted by date)")
print(f"{'='*60}")

# Store model in memory for next cells
print("\n✓ Model stored in variable 'model' for use in next cells")

In [0]:
# ============================================================================
# SHAP Explainability Analysis 
# ============================================================================

print("="*60)
print("SHAP EXPLAINABILITY ANALYSIS")
print("="*60)
print("\nComputing SHAP values...")
print("This may take 2-3 minutes...\n")

# ============================================================================
# PREPARE DATA FOR SHAP
# ============================================================================

# Use a sample of test data for SHAP (not all data)
sample_size = min(500, len(X_test))  # Use up to 500 samples, or less if X_test is smaller
print(f"Using {sample_size} samples from test set (out of {len(X_test)} total)")

# Sample from X_test (not from indices that might not exist)
X_shap = X_test.sample(n=sample_size, random_state=42)

print(f"✓ Sampled {len(X_shap)} records for SHAP analysis")

# ============================================================================
# COMPUTE SHAP VALUES
# ============================================================================

# Create SHAP explainer
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_shap)

print("✓ SHAP values computed")

# ============================================================================
# FEATURE IMPORTANCE FROM SHAP
# ============================================================================

# Calculate mean absolute SHAP values for feature importance
shap_importance = pd.DataFrame({
    'feature': feature_columns,
    'shap_importance': np.abs(shap_values).mean(axis=0)
}).sort_values('shap_importance', ascending=False)

print(f"\n{'='*60}")
print("SHAP FEATURE IMPORTANCE (Top 10)")
print(f"{'='*60}")
print("\nFeatures ranked by mean absolute SHAP value:")
print("(Higher = more impact on predictions)\n")

for idx, row in shap_importance.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['shap_importance']:.4f}")

# ============================================================================
# SHAP VISUALIZATION
# ============================================================================

print(f"\n{'='*60}")
print("Creating SHAP visualization...")

# Create bar plot of feature importance
fig_shap = go.Figure()

fig_shap.add_trace(go.Bar(
    x=shap_importance.head(15)['shap_importance'],
    y=shap_importance.head(15)['feature'],
    orientation='h',
    marker=dict(
        color=shap_importance.head(15)['shap_importance'],
        colorscale='Blues',
        showscale=True,
        colorbar=dict(title="SHAP<br>Value")
    )
))

fig_shap.update_layout(
    title="SHAP Feature Importance (Top 15 Features)",
    xaxis_title="Mean |SHAP Value|",
    yaxis_title="Feature",
    height=600,
    yaxis=dict(autorange="reversed")
)

fig_shap.show()

print("✓ SHAP visualization created")
print(f"{'='*60}")

# ============================================================================
# SAMPLE PATIENT EXPLANATION - FIXED
# ============================================================================

print(f"\nSample SHAP Explanation (One High-Risk Patient):")

# Get predictions for our SHAP sample
y_pred_shap = model.predict_proba(X_shap)[:, 1]

# Find highest risk patient in our SHAP sample
high_risk_idx = y_pred_shap.argmax()  # Index within X_shap (safe)
patient_shap = shap_values[high_risk_idx]  # Get SHAP values for this patient
patient_features = X_shap.iloc[high_risk_idx]  # Get feature values

print(f"Predicted sepsis probability: {y_pred_shap[high_risk_idx]:.3f}")
print(f"\nTop 5 features pushing prediction HIGHER:")

# Create feature contributions DataFrame
feature_contributions = pd.DataFrame({
    'feature': feature_columns,
    'value': patient_features.values,
    'shap': patient_shap
}).sort_values('shap', ascending=False)

for idx, row in feature_contributions.head(5).iterrows():
    print(f"  {row['feature']:25s} = {row['value']:6.2f}  (SHAP: +{row['shap']:.3f})")

print(f"\nTop 5 features pushing prediction LOWER:")
for idx, row in feature_contributions.tail(5).iterrows():
    print(f"  {row['feature']:25s} = {row['value']:6.2f}  (SHAP: {row['shap']:.3f})")

print(f"\n{'='*60}")
print("✓ SHAP analysis complete")
print(f"{'='*60}")

# Store SHAP results for potential future use
print("\n✓ SHAP values and importance stored in memory")
print("  - shap_values: SHAP values array")
print("  - shap_importance: Feature importance DataFrame")
print("  - X_shap: Sample data used for SHAP")

In [0]:
# ============================================================================
# Generate Early Warning Scores (EWS)
# ============================================================================

print("="*60)
print("GENERATING EARLY WARNING SCORES (EWS)")
print("="*60)

# Load all feature data
all_data = spark.table("healthcare_monitoring.icu_sepsis_features").toPandas()

# Generate predictions for all observations
print(f"\nScoring {len(all_data):,} observations...")

X_all = all_data[feature_columns]
all_proba = model.predict_proba(X_all)[:, 1]
all_pred = (all_proba >= 0.5).astype(int)

# Convert probabilities to EWS (0-100 scale)
ews_scores = (all_proba * 100).round(0).astype(int)

# Add to dataframe
all_data['sepsis_probability'] = all_proba
all_data['sepsis_predicted'] = all_pred
all_data['ews_score'] = ews_scores

# Categorize risk levels
def categorize_risk(score):
    if score >= 80:
        return "CRITICAL"
    elif score >= 60:
        return "HIGH"
    elif score >= 40:
        return "MEDIUM"
    else:
        return "LOW"

all_data['risk_category'] = all_data['ews_score'].apply(categorize_risk)

print("✓ Early Warning Scores generated")

# ============================================================================
# SUMMARY STATISTICS
# ============================================================================

print(f"\n{'='*60}")
print("EWS DISTRIBUTION")
print(f"{'='*60}")

risk_dist = all_data['risk_category'].value_counts()
print(f"\nRisk Category Distribution:")
for category in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']:
    count = risk_dist.get(category, 0)
    pct = 100 * count / len(all_data)
    print(f"  {category:10s}: {count:6,} ({pct:5.1f}%)")

print(f"\nEWS Score Statistics:")
print(f"  Mean:   {all_data['ews_score'].mean():.1f}")
print(f"  Median: {all_data['ews_score'].median():.1f}")
print(f"  Min:    {all_data['ews_score'].min()}")
print(f"  Max:    {all_data['ews_score'].max()}")

# High-risk patients
high_risk = all_data[all_data['ews_score'] >= 60]
print(f"\nHigh-risk observations (EWS ≥ 60): {len(high_risk):,}")
print(f"Unique patients at high risk: {high_risk['patient_id'].nunique()}")

# ============================================================================
# TOP HIGH-RISK OBSERVATIONS
# ============================================================================

print(f"\n{'='*60}")
print("TOP 10 HIGHEST RISK OBSERVATIONS")
print(f"{'='*60}")

top_risk = all_data.nlargest(10, 'ews_score')[
    ['patient_id', 'hour', 'ews_score', 'heart_rate', 'systolic_bp', 
     'lactate', 'sepsis_in_6h']
]
print(top_risk.to_string(index=False))

# ============================================================================
# SAVE TO DELTA TABLE
# ============================================================================

print(f"\n{'='*60}")
print("Saving EWS results to Delta table...")

result_df = spark.createDataFrame(all_data[[
    'patient_id', 'hour', 'timestamp',
    'sepsis_probability', 'sepsis_predicted', 'ews_score', 'risk_category',
    'sepsis_in_6h', 'has_sepsis_now'
]])

result_df.write.format("delta").mode("overwrite").saveAsTable(
    "healthcare_monitoring.sepsis_ews_results"
)

print("✓ Results saved to sepsis_ews_results table")

# ============================================================================
# VISUALIZATIONS
# ============================================================================

print(f"\nCreating EWS visualizations...")

# ============================================================================
# VISUALIZATION 1: EWS Score Histogram
# ============================================================================

fig1 = go.Figure()

fig1.add_trace(go.Histogram(
    x=all_data['ews_score'],
    nbinsx=50,
    marker_color='steelblue',
    name='EWS Scores'
))

fig1.update_layout(
    title="Early Warning Score (EWS) Distribution",
    xaxis_title="EWS Score (0-100)",
    yaxis_title="Count",
    height=400,
    showlegend=False
)

fig1.show()
print("✓ Histogram created")

# ============================================================================
# VISUALIZATION 2: Risk Category Pie Chart (Separate)
# ============================================================================

fig2 = go.Figure()

# Get risk distribution in correct order
risk_order = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
risk_counts = [risk_dist.get(cat, 0) for cat in risk_order]
colors = ['red', 'orange', 'yellow', 'lightgreen']

fig2.add_trace(go.Pie(
    labels=risk_order,
    values=risk_counts,
    marker=dict(colors=colors),
    textinfo='label+percent',
    textposition='inside'
))

fig2.update_layout(
    title="Risk Category Distribution",
    height=400
)

fig2.show()
print("✓ Pie chart created")

# ============================================================================
# VISUALIZATION 3: EWS Over Time (Sample Patient)
# ============================================================================

# Pick a patient who develops sepsis for interesting visualization
sepsis_patients = all_data[all_data['sepsis_in_6h'] == 1]['patient_id'].unique()

if len(sepsis_patients) > 0:
    sample_patient = sepsis_patients[0]
    patient_timeline = all_data[all_data['patient_id'] == sample_patient].sort_values('hour')
    
    fig3 = go.Figure()
    
    fig3.add_trace(go.Scatter(
        x=patient_timeline['hour'],
        y=patient_timeline['ews_score'],
        mode='lines+markers',
        name='EWS Score',
        line=dict(color='red', width=3),
        marker=dict(size=8)
    ))
    
    # Add threshold lines
    fig3.add_hline(y=80, line_dash="dash", line_color="red", 
                   annotation_text="CRITICAL", annotation_position="right")
    fig3.add_hline(y=60, line_dash="dash", line_color="orange", 
                   annotation_text="HIGH", annotation_position="right")
    fig3.add_hline(y=40, line_dash="dash", line_color="yellow", 
                   annotation_text="MEDIUM", annotation_position="right")
    
    fig3.update_layout(
        title=f"EWS Timeline - Patient {sample_patient}",
        xaxis_title="Hours Since ICU Admission",
        yaxis_title="Early Warning Score",
        height=400,
        yaxis_range=[0, 100]
    )
    
    fig3.show()
    print("✓ Timeline plot created")

# ============================================================================
# VISUALIZATION 4: Model Performance by EWS Threshold
# ============================================================================

# Calculate performance metrics at different EWS thresholds
thresholds = range(0, 101, 10)
performance = []

for thresh in thresholds:
    y_pred_thresh = (all_data['ews_score'] >= thresh).astype(int)
    y_true = all_data['sepsis_in_6h']
    
    if y_pred_thresh.sum() > 0 and y_true.sum() > 0:
        tp = ((y_pred_thresh == 1) & (y_true == 1)).sum()
        fp = ((y_pred_thresh == 1) & (y_true == 0)).sum()
        fn = ((y_pred_thresh == 0) & (y_true == 1)).sum()
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        performance.append({
            'threshold': thresh,
            'precision': precision,
            'recall': recall
        })

if performance:
    perf_df = pd.DataFrame(performance)
    
    fig4 = go.Figure()
    
    fig4.add_trace(go.Scatter(
        x=perf_df['threshold'],
        y=perf_df['precision'],
        mode='lines+markers',
        name='Precision',
        line=dict(color='blue', width=2)
    ))
    
    fig4.add_trace(go.Scatter(
        x=perf_df['threshold'],
        y=perf_df['recall'],
        mode='lines+markers',
        name='Recall',
        line=dict(color='green', width=2)
    ))
    
    fig4.update_layout(
        title="Model Performance by EWS Threshold",
        xaxis_title="EWS Threshold",
        yaxis_title="Score",
        height=400,
        yaxis_range=[0, 1]
    )
    
    fig4.show()
    print("✓ Performance plot created")

# ============================================================================
# PROJECT SUMMARY
# ============================================================================

print(f"\n{'='*60}")
print("✓ PROJECT 3 COMPLETE!")
print(f"{'='*60}")
print("\nKey Achievements:")
print("  • 1,000 ICU patient stays simulated")
print(f"  • {roc_auc:.3f} ROC-AUC on sepsis prediction")
print("  • 20 time-series features engineered")
print("  • SHAP explainability analysis complete")
print("  • Early Warning Scores (0-100) generated")
print(f"  • {len(high_risk):,} high-risk observations identified")
print(f"\nDelta Tables Created:")
print("  • healthcare_monitoring.icu_sepsis_data")
print("  • healthcare_monitoring.icu_sepsis_features")
print("  • healthcare_monitoring.sepsis_ews_results")
print(f"{'='*60}")
print("\n✓ Ready for dashboard creation and deployment!")
print(f"{'='*60}")